In [1]:
import gzip
import pickle
import pandas as pd

In [2]:
from mattergen.evaluation.reference.presets import ReferenceMP2020Correction

/home/kna/.conda/envs/vasp_computer/lib/python3.12/site-packages/mattergen/evaluation/reference/reference_dataset_serializer.py:19: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [4]:
alex_mp = ReferenceMP2020Correction.from_preset()

In [18]:
next(alex_mp.__iter__()) 

mp-1009460-GGA ComputedStructureEntry - Hf3          (Hf)
Energy (Uncorrected)     = -29.7388  eV (-9.9129  eV/atom)
Correction               = 0.0000    eV (0.0000   eV/atom)
Energy (Final)           = -29.7388  eV (-9.9129  eV/atom)
Energy Adjustments:
  None
Parameters:
  potcar_spec            = [{'titel': 'PAW_PBE Hf_pv 06Sep2000', 'hash': 'db924de35e43bb0854948d218ecbe730'}]
  is_hubbard             = False
  hubbards               = {}
  run_type               = GGA
Data:
  oxide_type             = None
  aspherical             = True
  last_updated           = 2023-07-28 09:44:06.902000
  task_id                = msft-2023072616422448636618438831
  material_id            = mp-1009460
  oxidation_states       = {}
  run_type               = GGA

In [ ]:
alex_mp.entries_by_chemsys["O"]

[mp-1009490-GGA ComputedStructureEntry - O2           (O2)
 Energy (Uncorrected)     = -9.8973   eV (-4.9487  eV/atom)
 Correction               = 0.0000    eV (0.0000   eV/atom)
 Energy (Final)           = -9.8973   eV (-4.9487  eV/atom)
 Energy Adjustments:
   None
 Parameters:
   potcar_spec            = [{'titel': 'PAW_PBE O 08Apr2002', 'hash': '7a25bc5b9a5393f46600a4939d357982'}]
   is_hubbard             = False
   hubbards               = {}
   run_type               = GGA
 Data:
   oxide_type             = None
   aspherical             = True
   last_updated           = 2023-07-28 01:28:35.307000
   task_id                = msft-2023072616540280271421435820
   material_id            = mp-1009490
   oxidation_states       = {}
   run_type               = GGA,
 mp-1023923-GGA ComputedStructureEntry - O6           (O2)
 Energy (Uncorrected)     = -23.9012  eV (-3.9835  eV/atom)
 Correction               = 0.0000    eV (0.0000   eV/atom)
 Energy (Final)           = -23.9012  eV (-

spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.


In [2]:
from matbench_discovery.data import DataFiles
with gzip.open(DataFiles.mp_patched_phase_diagram.mp_patched_phase_diagram.path, mode="rb") as zip_file:
    ppd_mp = pickle.load(zip_file)

In [3]:
from collections import Counter
Counter((entry.parameters['run_type'] for entry in ppd_mp.all_entries))

Counter({'GGA': 111762, 'GGA+U': 42956})

In [33]:
mp_20_csv_data = pd.read_csv("mp_20_test.csv.gz", index_col="material_id")
computed_data = pd.read_csv("mp_20_PBE_54_3.csv.gz", index_col="material_id")
computed_data['mp_20_csv_reference'] = mp_20_csv_data.reindex(computed_data.index).e_above_hull

In [34]:
from mp_api.client import MPRester
from dotenv import find_dotenv, get_key
MP_API_KEY = get_key(find_dotenv(), "MP_API_KEY")

In [35]:
with MPRester(MP_API_KEY) as mpr:
    for mp_id in computed_data.index:
        reference_records = mpr.get_entry_by_material_id(mp_id)
        for record in reference_records:
            if record.data['run_type'] in ("GGA", "GGA_U"):
                computed_data.loc[mp_id, "mp_e_above_hull"] = ppd_mp.get_e_above_hull(record, allow_negative=True)
                computed_data.loc[mp_id, "mp_e_uncorrected"] = record.uncorrected_energy

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/3 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/3 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/3 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
from pymatgen.entries.computed_entries import ComputedEntry
from ast import literal_eval
computed_data["PBE_54_energy_uncorrected"] = computed_data.entries.map(lambda x: ComputedEntry.from_dict(literal_eval(x)).uncorrected_energy)
computed_data.loc["mp-19449", "PBE_54_energy_uncorrected_sctipt"] = -82.51573047
computed_data.drop(columns=["entries", "structure"])

,e_above_hull_corrected,mp_20_csv_reference,mp_e_above_hull,mp_e_uncorrected,PBE_54_energy_uncorrected,PBE_54_energy_uncorrected_sctipt
material_id,,,,,,
mp-19449,0.045934,0.000000,-6.217249e-15,-83.158863,-82.515789,82.51573
mp-28599,-0.001522,0.000316,2.612457e-04,-23.581418,-23.593901,NaN
mp-1030119,-0.018732,0.002073,2.072728e-03,-90.783443,-91.033103,NaN
mp-1221948,0.131872,0.044558,4.455771e-02,-31.109103,-30.759844,NaN
mp-1103947,0.011350,0.000261,-7.124310e+00,-62.119994,-61.964743,NaN
mp-864930,0.022913,0.000000,0.000000e+00,-30.122192,-30.030538,NaN
mp-752737,0.159963,0.067626,6.763063e-02,-78.515757,-77.407773,NaN
mp-1516349,0.031824,0.028802,4.036693e-02,-74.691233,-74.776666,NaN
mp-1220086,0.135953,0.019049,-8.739883e+00,-92.506348,-91.103512,NaN
